In [1]:
%load_ext autoreload
%autoreload 2

import torch
from ncpu.nca import NeuralCA

from matplotlib import pyplot as plt
from IPython.display import clear_output, display
from tqdm.auto import tqdm

from ncpu.loss import loss_mse_whole_seq, loss_white_black, fullscreen_rollout_loss, output_masked_rollout_loss, combined_loss

model_path = None

In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())   # Should be True
print(torch.cuda.device_count())   # Should show 1
print(torch.cuda.get_device_name(0))  # Should show "NVIDIA GeForce GTX 1080 Ti"
torch.set_default_device('cuda')

In [3]:

from ncpu.config import TINY_AND_FARAWAY_TRAINING_CONFIG


LEARNING_RATE = 0.001
BATCH_SIZE = 8
GAUSSIAN_NOISE = 0.2
STEPS = 10_000

In [4]:
from ncpu.dataset import NCPUDataset 
dataset = NCPUDataset(TINY_AND_FARAWAY_TRAINING_CONFIG)


# Learn GATE Logic

In [6]:
from ncpu.trainer import NCPUTrainer

nca = NeuralCA(
    channels = 8,
    hidden_channels=[128],
    fire_rate = 0.99,
    alive_threshold=0.1,
    zero_initialization = False,
    kernel_size=5,
    read_only_dims=[-1]
)

trainer = NCPUTrainer(
    nca,
    dataset.get_dataloader(batch_size=BATCH_SIZE),
    lr=LEARNING_RATE,
    gaussian_noise=GAUSSIAN_NOISE,
    loss_fn=output_masked_rollout_loss,
)
trainer.sanity_check()
trainer.save_checkpoint()

In [7]:
pbar = tqdm(range(STEPS))
for i in pbar:
    info = trainer.optim_step(steps=(30, 80), return_rollout= (i % 100 == 0))
    loss = info["loss"]
    pbar.set_description(f"loss={loss:.6f}")

    if i % 100 == 0:
        clear_output(wait=False)
        display(pbar.container)

        trainer.display_optim_step(info)
        trainer.save_checkpoint()

trainer.save_checkpoint()

# Learning Information Propagation

In [ ]:
# load seed model
# trainer.load_checkpoint()

In [ ]:
stop_loss = 0.0001
pbar = tqdm(range(STEPS*4))
for i in pbar:
    info = trainer.optim_step(steps=(30, 80), loss=loss_mse_whole_seq)
    loss = info["loss"]
    pbar.set_description(f"loss={loss:.6f}")

    if i % 100 == 0:
        clear_output(wait=False)
        display(pbar.container)

        trainer.display_optim_step(info)
        trainer.save_checkpoint()

    if loss == stop_loss:
        break

trainer.save_checkpoint()

In [ ]:

trainer.save_checkpoint()

In [9]:
# load seed model
trainer.load_checkpoint()

In [ ]:
stop_loss = 0.0001
STEPS = 2000
pbar = tqdm(range(STEPS*10))
for i in pbar:
    info = trainer.optim_step(steps=(30, 80))
    loss = info["loss"]
    pbar.set_description(f"loss={loss:.6f}")

    if i % 100 == 0:
        clear_output(wait=False)
        display(pbar.container)

        trainer.display_optim_step(info)
        trainer.save_checkpoint()

    if loss == stop_loss:
        break

trainer.save_checkpoint()

In [ ]:
stop_loss = 0.0001
pbar = tqdm(range(1))
for i in pbar:
    info = trainer.optim_step(steps=(30, 80))
    loss = info["metrics"]["loss"]
    pbar.set_description(f"loss={loss:.6f}")

    clear_output(wait=False)
    display(pbar.container)

    trainer.display_optim_step(info)
